# Четвёртый этап: chunking и поиск по фрагментам

Ноутбук воспроизводит три стратегии разбиения документов SciFact, проверяет token budget BGE Small, показывает возобновляемую индексацию в Qdrant и сравнивает качество с документными baseline. 

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Находим корень проекта, даже если Jupyter был запущен из каталога notebooks.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден pyproject.toml')
os.chdir(PROJECT_ROOT)

def run_cli(*arguments: str) -> str:
    """Запустить проектный CLI тем же Python, что и kernel ноутбука."""
    result = subprocess.run(
        [sys.executable, '-m', 'rag_playground.cli', *arguments],
        cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
    )
    return result.stdout.strip()

print(f'Корень: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')

Корень: /Users/e.ilyina/Desktop/Портфолио/RAG
Python: 3.13.0


## 1. Параметры эксперимента

Параметры зафиксированы до просмотра test-метрик. В лимит входит и заголовок, и текст чанка.

In [2]:
from rag_playground.config import Settings

settings = Settings.from_env(PROJECT_ROOT / '.env')
safe_settings = {
    'embedding_model': settings.dense_model_name,
    'chunk_size': settings.chunk_size,
    'fixed_recursive_overlap_tokens': settings.chunk_overlap_tokens,
    'sentence_overlap': settings.chunk_sentence_overlap,
    'candidate_multiplier': settings.chunk_candidate_multiplier,
    'collection_prefix': settings.chunk_collection_prefix,
}
print(json.dumps(safe_settings, ensure_ascii=False, indent=2))

{
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "chunk_size": 256,
  "fixed_recursive_overlap_tokens": 32,
  "sentence_overlap": 1,
  "candidate_multiplier": 5,
  "collection_prefix": "scifact-chunks-bge-small-en-v1-5"
}


## 2. Подготовка трёх наборов чанков

Команда пересчитывает fixed, recursive и sentence-наборы. Она не обращается к платным API: используется локальный tokenizer BGE.

In [3]:
# В выводе для каждой стратегии должны быть нули в empty_chunks и over_limit_chunks.
prepared = json.loads(run_cli('chunk-prepare', '--strategy', 'all'))
summary = {
    name: {
        'chunks': item['report']['chunks'],
        'mean_tokens': item['report']['mean_tokens'],
        'max_tokens': item['report']['max_tokens'],
        'empty_chunks': item['report']['empty_chunks'],
        'over_limit_chunks': item['report']['over_limit_chunks'],
    }
    for name, item in prepared.items()
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "fixed": {
    "chunks": 9902,
    "mean_tokens": 204.449,
    "max_tokens": 256,
    "empty_chunks": 0,
    "over_limit_chunks": 0
  },
  "recursive": {
    "chunks": 9880,
    "mean_tokens": 202.94,
    "max_tokens": 256,
    "empty_chunks": 0,
    "over_limit_chunks": 0
  },
  "sentence": {
    "chunks": 10489,
    "mean_tokens": 200.387,
    "max_tokens": 256,
    "empty_chunks": 0,
    "over_limit_chunks": 0
  }
}


## 3. Как выглядят реальные чанки

Ниже берём первый документ, который дал несколько чанков, и показываем начало каждого фрагмента.

In [4]:
from collections import Counter

def read_jsonl(path: Path) -> list[dict]:
    """Прочитать небольшой исследуемый JSONL в список словарей."""
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]

chunks_dir = PROJECT_ROOT / 'data/processed/scifact/chunks'
fixed_rows = read_jsonl(chunks_dir / 'fixed.jsonl')
counts = Counter(row['doc_id'] for row in fixed_rows)
example_doc_id = next(doc_id for doc_id, count in counts.items() if count >= 3)
for strategy in ('fixed', 'recursive', 'sentence'):
    rows = [row for row in read_jsonl(chunks_dir / f'{strategy}.jsonl') if row['doc_id'] == example_doc_id]
    print(f'\n{strategy}: {len(rows)} чанков для doc_id={example_doc_id}')
    for row in rows[:3]:
        preview = row['text'][:180].replace('\n', ' ')
        print(f"  #{row['chunk_index']} — {row['token_count']} токенов — {preview}...")


fixed: 3 чанков для doc_id=87758
  #0 — 256 токенов — background common carotid intima media thickness ( cimt ) and ankle brachial pressure index ( abpi ) are used as surrogate marker of atherosclerosis, and have been shown to correla...
  #1 — 256 токенов — 31 anatomical arterial segments with each scored according to degree of luminal narrowing : 0 = normal, 1 = < 50 %, 2 = 50 - 70 %, 3 = 70 - 99 %, 4 = vessel occlusion. the segment ...
  #2 — 185 токенов — , which was due to a strong correlation with the ilio - femoral vessels with no correlation with the thoracic or neck vessels. on multiple linear regression, no correlation between...



recursive: 3 чанков для doc_id=87758
  #0 — 215 токенов — BACKGROUND Common carotid intima media thickness (CIMT) and ankle brachial pressure index (ABPI) are used as surrogate marker of atherosclerosis, and have been shown to correlate w...
  #1 — 240 токенов — volume acquisitions with a divided dose of intravenous gadolinium gadoterate meglumine ( dotarem, guerbet, fr ). The WB-MRA data was divided into 31 anatomical arterial segments wi...
  #2 — 242 токенов — its strong correlation with the neck and thoracic segments ( β 0. 42 p = 0. 01 ) with no correlation with the rest of the body. ABPI correlated with the whole body atheroma score (...

sentence: 4 чанков для doc_id=87758
  #0 — 215 токенов — BACKGROUND Common carotid intima media thickness (CIMT) and ankle brachial pressure index (ABPI) are used as surrogate marker of atherosclerosis, and have been shown to correlate w...
  #1 — 256 токенов — WB-MRA was performed in a 1.5T MRI scanner using 4 volume acquisitions with a divide

## 4. Возобновляемая индексация и состояние коллекций

Коллекции уже заполнены контрольным запуском. Поэтому повторный `chunk-index` для recursive должен пропустить все чанки и не пересчитывать embeddings. Затем `chunk-info` проверяет все три коллекции.

In [5]:
# Без --recreate существующая коллекция не удаляется.
print(run_cli('chunk-index', '--strategy', 'recursive'))

Collection 'scifact-chunks-bge-small-en-v1-5-recursive-s256' already exists; already indexed chunks will be skipped.
{
  "collection_name": "scifact-chunks-bge-small-en-v1-5-recursive-s256",
  "chunks_read": 9880,
  "chunks_skipped": 9880,
  "chunks_embedded": 0,
  "chunks_upserted": 0,
  "duration_seconds": 0.43358,
  "final_point_count": 9880
}


In [6]:
# Сверяем число точек Qdrant с числом чанков в соответствующем манифесте.
for strategy in ('fixed', 'recursive', 'sentence'):
    info = json.loads(run_cli('chunk-info', '--strategy', strategy))
    print(strategy, {
        'collection': info['collection_name'],
        'points': info['point_count'],
        'prepared_chunks': info['manifest']['chunks'],
        'status': info['status'],
    })

fixed {'collection': 'scifact-chunks-bge-small-en-v1-5-fixed-s256', 'points': 9902, 'prepared_chunks': 9902, 'status': 'green'}


recursive {'collection': 'scifact-chunks-bge-small-en-v1-5-recursive-s256', 'points': 9880, 'prepared_chunks': 9880, 'status': 'green'}


sentence {'collection': 'scifact-chunks-bge-small-en-v1-5-sentence-s256', 'points': 10489, 'prepared_chunks': 10489, 'status': 'green'}


## 5. Поиск по чанкам

Qdrant возвращает фрагменты, после чего код объединяет их по `doc_id`. Для каждого документа сохраняется лучший чанк — его удобно передавать генератору ответа на следующем этапе.

In [7]:
search_result = json.loads(run_cli(
    'chunk-search',
    'Vitamin D supplementation improves bone mineral density',
    '--strategy', 'recursive',
    '--top-k', '5',
))
print(f"Кандидатов-чанков: {search_result['candidate_chunks']}")
print(f"Полная задержка: {search_result['latency_ms']:.3f} мс")
for document in search_result['documents']:
    print(f"\n#{document['rank']} doc={document['doc_id']} score={document['score']:.4f}")
    print(f"Лучший чанк: {document['best_chunk_id']}")
    print(document['best_chunk_text'][:350])

Кандидатов-чанков: 25
Полная задержка: 16.430 мс

#1 doc=38551172 score=0.8036
Лучший чанк: 38551172:recursive:0000
Mammographic density is a strong risk factor for breast cancer, but the underlying biology for this association is unknown. Studies suggest that vitamin D may reduce breast cancer risk and dietary vitamin D intake has been associated with reduced breast density. We conducted a case-control study nested within the Nurses' Health Study cohort consist

#2 doc=16256507 score=0.7956
Лучший чанк: 16256507:recursive:0000
CONTEXT Adequate vitamin D status for optimum bone health has received increased recognition in recent years; however, the ideal intake is not known. Serum 25-hydroxyvitamin D is the generally accepted indicator of vitamin D status, but no universal reference level has been reached. OBJECTIVE To investigate the relative importance of high calcium i

#3 doc=13312471 score=0.7890
Лучший чанк: 13312471:recursive:0000
Vitamin D insufficiency is common, however withi

## 6. Итоговое сравнение

Пакетные ranking-файлы уже построены для 300 запросов. Команда заново считает единые метрики по неизменённому test qrels. Test split используется только для финального отчёта, а не для подбора параметров.

In [8]:
comparison = json.loads(run_cli('chunk-compare', '--split', 'test', '--top-k', '100'))
columns = ('recall@1', 'recall@10', 'mrr@10', 'ndcg@10', 'recall@100')
print('method'.ljust(24), *(column.rjust(11) for column in columns))
for method, values in comparison['metrics'].items():
    rendered = [f"{values[column]:.4f}".rjust(11) for column in columns]
    print(method.ljust(24), *rendered)
print('\n', comparison['note'])

method                      recall@1   recall@10      mrr@10     ndcg@10  recall@100
bm25_document                 0.5282      0.8173      0.6464      0.6826      0.9282
dense_document                0.5759      0.8362      0.6822      0.7127      0.9417
hybrid_document               0.5718      0.8612      0.6910      0.7268      0.9683
chunk_fixed                   0.5701      0.8386      0.6802      0.7143      0.9517
chunk_recursive               0.5984      0.8340      0.6951      0.7249      0.9517
chunk_sentence                0.5859      0.8336      0.6882      0.7178      0.9550

 Configurations are fixed in advance; test metrics are not used for tuning.


## 7. Автоматические проверки

Быстрые тесты проверяют границы чанков, overlap, стабильные ID, агрегацию и сохранение артефактов. Интеграционный тест Qdrant запускается отдельно, когда локальный сервис доступен.

In [9]:
# Ruff проверяет стиль и типичные ошибки; Pytest — поведение функций.
ruff = subprocess.run(
    [sys.executable, '-m', 'ruff', 'check', 'src', 'tests'],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
)
tests = subprocess.run(
    [sys.executable, '-m', 'pytest'],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
)
print('Ruff:', ruff.stdout.strip())
print('Pytest:', tests.stdout.strip())

Ruff: All checks passed!
Pytest: sss...................................                                   [100%]
35 passed, 3 skipped in 1.44s


## Вывод

Все 5 183 документов представлены токен-безопасными чанками. Recursive оказался лучшим из трёх chunk-вариантов по nDCG@10, однако документный hybrid сохранил лучший Recall@10. На пятом этапе найденные лучшие чанки станут контекстом для генерации RAG-ответов.